In [ ]:
words = open('../names.txt', 'r').read().splitlines()

In [ ]:
words[:10]

In [ ]:
len(words)

In [ ]:
min(len(w) for w in words)

In [ ]:
max(len(w) for w in words)

In a bigram language model, we're only ever looking at 2 characters at a time. 

So we're looking at 1 character we are given, and trying to predicted the next character in a sequence.

We're ignoring the fact that we may have a lot more information. It's a very simple and weak language model.

In [ ]:
b = {}
for w in words:
    chs = ['<S>'] + list(w) + ['<E>']
    for ch1, ch2 in zip(chs, chs[1:]):
        bigram = (ch1, ch2)
        b[bigram] = b.get(bigram, 0) + 1

In [ ]:
sorted(b.items(), key = lambda kv: -kv[1])

In [ ]:
import torch

In [ ]:
N = torch.zeros((27, 27), dtype=torch.int32)    # Bigram counts
                                                # 26 alphabet characters + 1 special character

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [ ]:

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(16,16))
plt.imshow(N, cmap='Blues')
for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color='black')
        plt.text(j, i, N[i, j].item(), ha="center", va="top", color='black')
plt.axis('off')

In [ ]:
N[0]

Now convert raw counts to probabilities.

In [ ]:
p = N[0].float() # get all characters in first row (count of words that start with each char)
p = p / p.sum() # probability of any character to be the first character of a word; sums to 1
p

Now we can sample from this distribution

In [ ]:
# workaround to get the same results as the video, which uses an older pytorch
def sample(p, g):
    # replicates old torch.multinomial(p, 1) so results match the video
    u = torch.rand(1, generator=g, dtype=torch.float64)
    return int(torch.searchsorted(p.double().cumsum(0), u))

In [ ]:
g = torch.Generator().manual_seed(2147483647)
ix = sample(p,g)
itos[ix]

In [ ]:
g = torch.Generator().manual_seed(2147483647) # Generator makes everything deterministic so I get the same results as Karpathy's tutorial
p = torch.rand(3, generator=g)
p = p / p.sum()
p

In [ ]:
torch.multinomial(p, num_samples=20, replacement=True, generator=g) # torch.multinomial draws samples: 
                                                                    # takes the tensor of probability distributions (p)

In [ ]:
p.shape

In [ ]:
P.shape

In [ ]:
P.sum(1, keepdim=True).shape

In [ ]:
# Let's make the below block more efficient by making the P probability distribution matrix
# (instead of calculating it every loop)
P = N.float()
P /= P.sum(1, keepdim=True) # normalize by column vec (sum all rows and normalize) (normalize rows)

# P is really the parameters of our bigram language model. It summarizes the statistics of our bigrams.

In [ ]:
P[0].sum()

In [ ]:
g = torch.Generator().manual_seed(2147483647)

for i in range(5):
    
    out = []
    ix = 0
    while True:
        
        p = P[ix]
        #p = N[ix].float() 
        #p = p / p.sum() # convert count to probability distribution
        #p = torch.ones(27) / 27.0 # sample from uniform distribution to compare (completely untrained model)
        
        ix = sample(p,g) # sample distribution to get next char (row index)
        out.append(itos[ix]) # append sampled next char to out list
        if ix == 0: # if end token: end word
            break
    print(''.join(out))

Next, let's try to evaluate the quality of the model.

We want to summarize the quality of the model into a single number: how good is it at predicting, let's say the training set.

So now in the training set we can evaluate the training loss, which tells us about the quality of this model in a single number (just like in micrograd).